In [10]:
%load_ext autoreload
%autoreload 2

In [11]:
import abc, gc, os, random, re, sys, time, traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stable_baselines3 as sb
from stable_baselines3.common.env_util import make_vec_env
from typing import SupportsFloat, Any
import gymnasium as gym
from gymnasium.core import ActType, ObsType

import faiss
import sklearn
from tqdm import tqdm
import chime
import scipy.sparse as sp

import torch
from torch.nn import functional as F
from torch.utils import data
from torch import optim

In [12]:
from data_loaders import KuaiRecData, commit_dataset, CommitedDataset, JaggedCommittedDataset, commit_dataset_jagged
from dl_trainer import Trainer, sequence_collate, create_sequence_collator
from rs_algorithm.models import AdaptiveRepresentation
from rs_algorithm.models import TwoTowerRecommender, ThreeTowerRecommenderV2, ConcatInteractionHistoryRepresentationWithUserEmbedInitialization
from rs_algorithm.models.user_models.user_02_sim import DynamicPreferenceUserModel, norm_dot_product_based_relavance_calculation, SimUserModel

In [13]:
from implicit.als import AlternatingLeastSquares

In [161]:
data_model = KuaiRecData(root='../data/KuaiRec2.0', initial_specimen='smaller')

In [227]:
interactions = data_model.ratings[['user_id', 'item_id', 'rating']]

user_ids = interactions['user_id'].to_numpy(dtype=np.int64)
item_ids = interactions['item_id'].to_numpy(dtype=np.int64)
ratings = interactions['rating'].to_numpy(dtype=np.float32)
ratings = rating = 1 + 40 * ratings

In [228]:
def get_dimensions(user_ids, item_ids):
    num_users = int(user_ids.max()) + 1
    num_items = int(item_ids.max()) + 1
    return num_users, num_items

In [229]:
def train_test_split(user_ids, item_ids, ratings):
    order = np.argsort(user_ids)

    user_ids = user_ids[order]
    item_ids = item_ids[order]
    ratings = ratings[order]

    train_u, train_i, train_r = [], [], []
    test = {}

    current_user = None
    buffer = []

    for u, i, r in zip(user_ids, item_ids, ratings):
        if current_user is None:
            current_user = u

        if u != current_user:
            if len(buffer) > 0:
                *train_part, test_item = buffer
                for x in train_part:
                    train_u.append(x[0])
                    train_i.append(x[1])
                    train_r.append(x[2])

                test[int(current_user)] = int(test_item[1])

            buffer = []
            current_user = u

        buffer.append((int(u), int(i), float(r)))

    # last user
    if len(buffer) > 0:
        *train_part, test_item = buffer
        for x in train_part:
            train_u.append(x[0])
            train_i.append(x[1])
            train_r.append(x[2])

        test[int(current_user)] = int(test_item[1])

    return (
        np.array(train_u, dtype=np.int64),
        np.array(train_i, dtype=np.int64),
        np.array(train_r, dtype=np.float32),
        test
    )

In [230]:
def build_sparse_matrix(user_ids, item_ids, ratings, num_users, num_items):
    return sp.csr_matrix(
        (ratings, (user_ids, item_ids)),
        shape=(num_users, num_items),
        dtype=np.float32
    )

In [231]:
def train_als(train_matrix, factors=64, regularization=0.01, iterations=15):
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations
    )

    model.fit(train_matrix)
    return model

In [232]:
def get_recommendations(model, user_id, train_matrix, N=10):
    user_items = train_matrix[user_id]

    item_ids, scores = model.recommend(
        userid=int(user_id),
        user_items=user_items,
        N=N,
        filter_already_liked_items=True
    )

    return item_ids.tolist()

In [233]:
def evaluate_model(model, train_matrix, test, K=10):
    recs = {}

    for u in test.keys():
        u_int = int(u)

        if train_matrix[u_int].nnz == 0:
            continue

        recs[u_int] = get_recommendations(model, u_int, train_matrix, N=K)

    return {
        "HR@K": hit_rate_at_k(recs, test, K),
        "Precision@K": precision_at_k(recs, test, K),
        "Recall@K": recall_at_k(recs, test, K),
        "NDCG@K": ndcg_at_k(recs, test, K),
    }

In [234]:
def hit_rate_at_k(recs, test, K):
    return np.mean([
        1 if test[u] in recs[u][:K] else 0
        for u in test
    ])

In [235]:
def precision_at_k(recs, test, K):
    return np.mean([
        (1 if test[u] in recs[u][:K] else 0) / K
        for u in test
    ])

In [236]:
def recall_at_k(recs, test, K):
    return np.mean([
        1 if test[u] in recs[u][:K] else 0
        for u in test
    ])

In [237]:
def ndcg_at_k(recs, test, K):
    scores = []

    for u in test:
        gt = test[u]
        user_recs = recs[u][:K]

        if gt in user_recs:
            rank = user_recs.index(gt) + 1
            dcg = 1 / np.log2(rank + 1)
        else:
            dcg = 0

        scores.append(dcg)  # IDCG = 1

    return np.mean(scores)

In [238]:
def evaluate_model(model, train_matrix, test, K=10):
    recs = {}

    for u in tqdm(test.keys()):
        recs[u] = get_recommendations(model, u, train_matrix, N=K)

    return {
        "HR@K": hit_rate_at_k(recs, test, K),
        "Precision@K": precision_at_k(recs, test, K),
        "Recall@K": recall_at_k(recs, test, K),
        "NDCG@K": ndcg_at_k(recs, test, K),
    }

In [258]:
num_users, num_items = get_dimensions(user_ids, item_ids)

train_u, train_i, train_r, test_data = train_test_split(user_ids, item_ids, ratings)
train_matrix = build_sparse_matrix(train_u, train_i, train_r, num_users, num_items)
model = train_als(train_matrix, 64, 0.01, 10)

metrics = evaluate_model(model, train_matrix, test_data, K=10)

print(metrics)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 1411/1411 [00:00<00:00, 4910.59it/s]

{'HR@K': np.float64(0.002834868887313962), 'Precision@K': np.float64(0.0002834868887313962), 'Recall@K': np.float64(0.002834868887313962), 'NDCG@K': np.float64(0.001591879117547216)}


In [244]:
# rating = 1 + 40 * rating
{'HR@K': np.float64(0.0063784549964564135), 'Precision@K': np.float64(0.0006378454996456414), 'Recall@K': np.float64(0.0063784549964564135), 'NDCG@K': np.float64(0.003571225157362052)}

{'HR@K': np.float64(0.0063784549964564135),
 'Precision@K': np.float64(0.0006378454996456414),
 'Recall@K': np.float64(0.0063784549964564135),
 'NDCG@K': np.float64(0.003571225157362052)}

In [240]:
# float
{'HR@K': np.float64(0.004252303330970942), 'Precision@K': np.float64(0.0004252303330970943), 'Recall@K': np.float64(0.004252303330970942), 'NDCG@K': np.float64(0.0016921979037703707)}

{'HR@K': np.float64(0.004252303330970942),
 'Precision@K': np.float64(0.0004252303330970943),
 'Recall@K': np.float64(0.004252303330970942),
 'NDCG@K': np.float64(0.0016921979037703707)}

In [223]:
# 1000 * int
{'HR@K': np.float64(0.002834868887313962), 'Precision@K': np.float64(0.0002834868887313962), 'Recall@K': np.float64(0.002834868887313962), 'NDCG@K': np.float64(0.0020770209483156575)}

{'HR@K': np.float64(0.002834868887313962),
 'Precision@K': np.float64(0.0002834868887313962),
 'Recall@K': np.float64(0.002834868887313962),
 'NDCG@K': np.float64(0.0020770209483156575)}

In [224]:
num_users, num_items

(7163, 10596)

In [225]:
model.user_factors.shape, model.item_factors.shape

((7163, 64), (10596, 64))